# preprocessing line-by-line test

`src/utils/preprocessing.py`의 핵심 로직을 셀 단위로 한 줄씩 검증하는 테스트 노트북입니다.


In [ ]:
import os
import re
import pandas as pd
from glob import glob

from src.utils.preprocessing import Preprocessor

negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', 'Rule out']
abbrev_path = os.path.join('.', 'src', 'utils', 'abbrevations.json')

raw_data_dir = glob(os.path.join('data','reports','*'))
print('raw_data_dir:', raw_data_dir)

df = pd.read_csv(raw_data_dir[0], encoding='utf-8-sig', engine='c').reset_index()
text_df = df['검사결과결론내용'].dropna()

pre = Preprocessor(
    df=text_df,
    negative_patterns=negative_patterns,
    uncertain_patterns=uncertain_patterns,
    abbrev_path=abbrev_path,
)

print('loaded rows:', len(text_df))


## 1) `__init__` + 약어 확장(`df.apply(self.expand_abbrev_value)`) 검증


In [ ]:
assert isinstance(pre.abbrev_dict, dict)
assert hasattr(pre, 'regex') and hasattr(pre, 'negative_regex') and hasattr(pre, 'uncertain_regex')
assert len(pre.df) == len(text_df)

# 비문자 입력 처리 동작도 같은 인스턴스에서 확인
assert pre.expand_abbrev_value(123) == 123
assert pre.expand_abbrev_value(None) is None

print('init ok (real data)')


## 2) `replace_abbrev` 단일 매칭 검증


In [ ]:
if pre.abbrev_dict:
    some_key = next(iter(pre.abbrev_dict.keys()))
    expected = pre.abbrev_dict[some_key]
    m = re.search(rf'(?<!\S)({re.escape(some_key)})(?!\S)', some_key)
    assert m is not None
    got = pre.replace_abbrev(m)
    assert got == expected
    print('replace_abbrev ok:', some_key, '->', got)
else:
    print('abbrev_dict is empty: skip')


## 3) `expand_abbrev_value` 검증 (문자/비문자/개행 정리)


In [ ]:
assert pre.expand_abbrev_value(3.14) == 3.14  # not str -> 그대로 반환
assert pre.expand_abbrev_value(None) is None

cleaned = pre.expand_abbrev_value('line1\n    \nline2\n----- > x')
assert '\n    \n' not in cleaned
assert '----- >' not in cleaned
print('expand_abbrev_value ok:', repr(cleaned))


## 4) `word_patterns` / 패턴 카운트 검증


In [ ]:
pattern_text = Preprocessor.word_patterns(['no', 'negative'])
regex = re.compile(pattern_text, re.IGNORECASE)

assert Preprocessor.count_patterns('No metastasis. negative node.', regex) >= 2
assert Preprocessor.count_patterns(None, regex) == 0
print('word_patterns/count_patterns ok:', pattern_text)


## 5) `extract_parts` 검증


In [ ]:
s = '-1. metastasis found\n2. no recurrence\n3) rule out recurrence'
parts = Preprocessor.extract_parts(s, 'recur')
assert len(parts) >= 1
print('extract_parts ok:', parts)


## 6) `count_list` 검증


In [ ]:
test_list = ['no metastasis', 'rule out recurrence', 'metastasis suspected']
counts = pre.count_list(test_list)
assert len(counts) == 3
assert counts[0] == len(test_list)
print('count_list ok:', counts)


## 7) `target_filtering` end-to-end 검증


In [ ]:
target_text = 'metastasis'
result = pre.target_filtering(target_text)

assert set(result.keys()) == {'positive', 'negative', 'uncertain'}
print('target_filtering keys:', list(result.keys()))

for k, v in result.items():
    print('---', k, '| count =', len(v))
    if len(v) > 0:
        print(v.iloc[0][:300])


## 사용 방법

위 셀을 위에서 아래로 실행하면 `preprocessing.py` 주요 로직을 단계별로 점검할 수 있습니다.
실데이터에 맞춘 케이스를 추가하고 싶으면 각 섹션에 assert를 더 붙여 확장하세요.


In [ ]:
import re
import pandas as pd
from src.utils.preprocessing import Preprocessor


def debug_single_text_full(
    T: str,
    negative_patterns,
    uncertain_patterns,
    abbrev_path="./src/utils/abbrevations.json",
    targets=("recur", "metas"),
):
    """
    단일 문장 T에 대해 Preprocessor 파이프라인을 최대한 세분화해서 기록.
    반환:
      1) step_df: 전처리 단계별 텍스트 변화
      2) split_df: extract_parts의 split 결과(구분자/조각 포함)
      3) target_df: target별로 어떤 조각이 살아남았고 어떤 라벨이 되었는지
      4) tf_result: pre.target_filtering(target) 실제 결과(호출 가능한 경우만)
    """
    # 실제 구현과 동일하게 Preprocessor 초기화
    pre = Preprocessor(
        df=pd.Series([T]),
        negative_patterns=negative_patterns,
        uncertain_patterns=uncertain_patterns,
        abbrev_path=abbrev_path,
    )

    # expand_abbrev_value 내부 단계 분해
    x0 = T
    x1 = pre.regex.sub(pre.replace_abbrev, x0) if isinstance(x0, str) else x0
    x2 = re.sub(r"\n[ \t\n]+", "\n", x1) if isinstance(x1, str) else x1
    x3 = re.sub(r"\n[\=\-\~#]+>?\s?", "\n", x2) if isinstance(x2, str) else x2
    x4 = re.sub(r"\n[\=\-\~#]+>", "", x3) if isinstance(x3, str) else x3  # final normalized

    step_df = pd.DataFrame([
        {"step": "0_raw", "text": x0},
        {"step": "1_expand_abbrev", "text": x1},
        {"step": "2_normalize_newlines", "text": x2},
        {"step": "3_remove_sep_a", "text": x3},
        {"step": "4_remove_sep_b(final)", "text": x4},
    ])

    # extract_parts 내부 split 과정까지 그대로 노출
    split_pattern = r"(^\-\d+\.\s?|\n\d+\.\s?|\n\d+\))"
    split_rows = []

    if isinstance(x4, str):
        raw_split = re.split(split_pattern, x4)  # delimiter도 포함됨
        # filter(None, ...) 적용 전/후를 모두 기록
        for i, piece in enumerate(raw_split):
            split_rows.append({
                "order": i,
                "piece_raw": piece,
                "is_empty": (piece == "" or piece is None),
                "piece_after_filter_none": piece if piece else None,
                "looks_like_delimiter": bool(re.fullmatch(split_pattern, piece or "")),
            })
    else:
        raw_split = []

    split_df = pd.DataFrame(split_rows)

    target_rows = []
    tf_result = {}

    for target in targets:
        target_l = target.lower().strip()

        if not isinstance(x4, str):
            target_rows.append({
                "target": target_l,
                "contains_target_in_full_text": False,
                "parts_kept_by_extract_parts": [],
                "parts_all_non_empty": [],
                "n_parts_total_non_empty": 0,
                "n_parts_kept": 0,
                "neg_count_on_kept_parts": 0,
                "unc_count_on_kept_parts": 0,
                "label_by_target_filtering_logic": "dropped_non_string",
            })
            tf_result[target_l] = {
                "positive": pd.Series(dtype="object"),
                "negative": pd.Series(dtype="object"),
                "uncertain": pd.Series(dtype="object"),
            }
            continue

        contains_target = target_l in x4.lower()

        # extract_parts 로직과 동일
        parts_all_non_empty = list(filter(None, raw_split))
        parts_kept = [p for p in parts_all_non_empty if target_l in p.lower()]

        # count_list와 동일
        cnt_total, cnt_neg, cnt_unc = pre.count_list(parts_kept)

        # target_filtering 라벨 로직 동일
        unc_idx = (cnt_unc > 0)
        pos_idx = (cnt_total > cnt_neg)
        pos_idx = ((pos_idx ^ unc_idx) & pos_idx)
        neg_idx = not (pos_idx or unc_idx)

        if not contains_target:
            label = "dropped_no_target"
        elif pos_idx:
            label = "positive"
        elif unc_idx:
            label = "uncertain"
        else:
            label = "negative"

        target_rows.append({
            "target": target_l,
            "contains_target_in_full_text": contains_target,
            "parts_all_non_empty": parts_all_non_empty,           # 스플릿 후 살아있는 전체 조각
            "parts_kept_by_extract_parts": parts_kept,            # 타겟 포함 조각만
            "n_parts_total_non_empty": len(parts_all_non_empty),
            "n_parts_kept": cnt_total,
            "neg_count_on_kept_parts": cnt_neg,
            "unc_count_on_kept_parts": cnt_unc,
            "label_by_target_filtering_logic": label,
        })

        # ValueError 방지: 타겟이 실제 텍스트에 있을 때만 호출
        if contains_target:
            tf_result[target_l] = pre.target_filtering(target_l)
        else:
            tf_result[target_l] = {
                "positive": pd.Series(dtype="object"),
                "negative": pd.Series(dtype="object"),
                "uncertain": pd.Series(dtype="object"),
            }

    target_df = pd.DataFrame(target_rows)
    return step_df, split_df, target_df, tf_result

In [ ]:
T = text_df.sample(1).item()
print(T)

step_df, split_df, target_df, tf_result = debug_single_text_full(
    T=T,
    negative_patterns=negative_patterns,
    uncertain_patterns=uncertain_patterns,
    targets=("recur", "metas"),
)


In [ ]:
print(step_df.iloc[4]['text'])

In [ ]:
split_df

In [ ]:
target_df

In [ ]:
print(step_df['text'].iloc[2])

In [ ]:
step_df['text'].iloc[0]